# Self-Attention (自注意力) 实现

自注意力是Transformer的核心，让序列中每个位置都能关注所有位置。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

## 实现自注意力类

In [ ]:
class SelfAttention:
    def __init__(self, embed_dim, use_bias=True):
        self.embed_dim = embed_dim
        self.use_bias = use_bias
        
        # Q、K、V投影矩阵
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(embed_dim)
            self.b_v = np.zeros(embed_dim)
    
    def forward(self, x, mask=None):
        # 线性投影
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # Scaled Dot-Product Attention
        scores = np.dot(Q, K.T) / np.sqrt(self.embed_dim)
        
        # 应用mask
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        
        # Softmax
        attention_weights = softmax(scores, axis=-1)
        
        # 加权求和
        output = np.dot(attention_weights, V)
        
        return output, attention_weights

## 测试自注意力

In [ ]:
seq_len = 8
embed_dim = 64

x = np.random.randn(seq_len, embed_dim)
self_attn = SelfAttention(embed_dim)

output, attn_weights = self_attn.forward(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {attn_weights.shape}")

## 可视化注意力权重矩阵

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(attn_weights, annot=True, fmt='.2f', cmap='YlOrRd')
plt.xlabel('Key位置')
plt.ylabel('Query位置')
plt.title('自注意力权重矩阵')
plt.show()

## 因果Mask（GPT风格）

In [ ]:
def create_causal_mask(seq_len):
    return np.tril(np.ones((seq_len, seq_len)))

causal_mask = create_causal_mask(seq_len)
print("因果Mask:")
print(causal_mask.astype(int))

output_masked, attn_weights_masked = self_attn.forward(x, mask=causal_mask)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(causal_mask, annot=True, fmt='.0f', cmap='Greys', ax=axes[0], cbar=False)
axes[0].set_title('因果Mask')
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

sns.heatmap(attn_weights_masked, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('带Mask的注意力权重')
axes[1].set_xlabel('Key位置')
axes[1].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

## 分析单个位置的注意力分布

In [ ]:
position = 5
attention_at_pos = attn_weights[position]

plt.figure(figsize=(10, 4))
plt.bar(range(seq_len), attention_at_pos)
plt.xlabel('位置索引')
plt.ylabel('注意力权重')
plt.title(f'位置 {position} 对所有位置的注意力分布')
plt.grid(True, alpha=0.3)
plt.show()

print(f"最关注的位置: {np.argmax(attention_at_pos)}")
print(f"对自己的注意力: {attention_at_pos[position]:.4f}")